# Saving Your Clean Data

**Topic:** Data Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import os
import joblib

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import IntSlider, Output, VBox, HTML
from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer

import seaborn as sns
titanic = sns.load_dataset("titanic")

from tkh_utils import PALETTE, FONT, base_layout

np.random.seed(42)

---
## What you'll explore

By the end of this notebook you will be able to:

- **Describe** the complete preprocessing sequence from raw Titanic data to model-ready train/test splits
- **Explain** why we save the fitted scaler alongside the data files
- **Interpret** the purpose of each saved file and how they will be used in the model folder

> **Tip:** Step through each stage in the widget below. Watch the shape change, the missing values drop to zero, and the numeric columns shift to a common scale.

---
## How we got here

In `09_cross_validation_and_preprocessing.ipynb` we saw that preprocessing must happen inside each fold, and that `sklearn.Pipeline` handles this automatically in the model folder.

Now we complete the preprocessing folder's job: apply every step we have learned to the Titanic dataset in the correct order, and save the clean result so the model folder can pick it up. This is the handoff between the preprocessing phase and the modeling phase.

---
## Why this matters for data science

In real projects, preprocessing and modeling are often done by different people or at different times. Saving clean, versioned train/test files as artifacts means:
- Anyone who opens the model folder gets the same data
- The test set stays locked and cannot be accidentally contaminated
- The fitted scaler travels with the data, so new predictions can be transformed the same way

---
## Try it yourself: step through the preprocessing workflow

In [ ]:
out = Output()

# Build the complete preprocessing pipeline step by step
# so we can inspect the data at each stage.

# ── Step 0: Raw data ────────────────────────────────────────────────────────
raw = titanic[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "survived"]].copy()

# ── Step 1: Binary indicator for cabin ──────────────────────────────────────
s1 = raw.copy()
s1["has_cabin"] = titanic["cabin"].notna().astype(int)

# ── Step 2: Impute missing values ────────────────────────────────────────────
s2 = s1.copy()
s2["age"]      = s2["age"].fillna(s2["age"].median())
s2["embarked"] = s2["embarked"].fillna(s2["embarked"].mode()[0])
s2["fare"]     = s2["fare"].fillna(s2["fare"].median())

# ── Step 3: Encode categorical columns ──────────────────────────────────────
s3 = pd.get_dummies(s2, columns=["sex", "embarked"], drop_first=False)

# ── Step 4: Train-test split ─────────────────────────────────────────────────
X_raw_split = s3.drop(columns="survived")
y_raw_split = s3["survived"]

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_raw_split, y_raw_split, test_size=0.20, random_state=42, stratify=y_raw_split
)

# ── Step 5: Scale — fit on train, transform both ─────────────────────────────
numeric_cols = ["pclass", "age", "sibsp", "parch", "fare"]
scaler = StandardScaler()

X_tr_final = X_tr_raw.copy()
X_te_final = X_te_raw.copy()

X_tr_final[numeric_cols] = scaler.fit_transform(X_tr_raw[numeric_cols])
X_te_final[numeric_cols] = scaler.transform(X_te_raw[numeric_cols])

STEPS = [
    {
        "title": "Step 0: Raw Titanic data",
        "df": raw.drop(columns="survived").head(6),
        "shape": raw.shape,
        "note": "Raw data as loaded. Contains text, missing values, and features on different scales.",
        "missing": raw.isna().sum().sum(),
    },
    {
        "title": "Step 1: Add binary cabin indicator",
        "df": s1.drop(columns="survived").head(6),
        "shape": s1.shape,
        "note": "cabin column is 77% missing: we extract the signal, was a cabin recorded at all?",
        "missing": s1.isna().sum().sum(),
    },
    {
        "title": "Step 2: Impute missing values",
        "df": s2.drop(columns="survived").head(6),
        "shape": s2.shape,
        "note": "age filled with median (28.0), embarked filled with mode ('S'), fare filled with median.",
        "missing": s2.isna().sum().sum(),
    },
    {
        "title": "Step 3: Encode categorical columns",
        "df": s3.drop(columns="survived").head(6),
        "shape": s3.shape,
        "note": "sex and embarked one-hot encoded. No text columns remain.",
        "missing": 0,
    },
    {
        "title": "Step 4: Train-test split (80/20, stratified)",
        "df": X_tr_raw.head(6),
        "shape": (len(X_tr_raw), X_tr_raw.shape[1]),
        "note": f"Training set: {len(X_tr_raw)} rows. Test set: {len(X_te_raw)} rows. Stratified on survived.",
        "missing": 0,
    },
    {
        "title": "Step 5: Scale numeric features (train only fit)",
        "df": X_tr_final[numeric_cols].head(6).round(3),
        "shape": X_tr_final.shape,
        "note": "StandardScaler fit on training rows only. Same scaler then transforms test rows.",
        "missing": 0,
    },
]

step_slider = IntSlider(
    value=0, min=0, max=len(STEPS) - 1, step=1,
    description="Step:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="500px"),
)

def render(change=None):
    s = STEPS[step_slider.value]
    df_display = s["df"].reset_index(drop=True)
    n_rows, n_cols = df_display.shape

    # Table of first 6 rows
    fig = go.Figure(go.Table(
        header=dict(
            values=[f"<b>{c}</b>" for c in df_display.columns],
            fill_color=PALETTE["primary"],
            font=dict(color="white", size=11),
            align="left",
        ),
        cells=dict(
            values=[df_display[c].tolist() for c in df_display.columns],
            fill_color=[
                ["#ffffff" if r % 2 == 0 else PALETTE["background"] for r in range(n_rows)]
                for _ in range(n_cols)
            ],
            align="left",
            font=dict(size=11),
        ),
    ))
    layout = base_layout(title=s["title"])
    layout.update(height=300, margin=dict(l=20, r=20, t=60, b=10))

    missing_color = PALETTE["secondary"] if s["missing"] > 0 else PALETTE["accent"]
    info_html = (
        f'<div style="font-family:Inter,Arial,sans-serif;max-width:620px;">'
        f'<div style="display:flex;gap:16px;margin-bottom:8px;">'
        f'<span style="background:{PALETTE["primary"]};color:#fff;padding:4px 10px;'
        f'border-radius:12px;font-size:12px;">Shape: {s["shape"]}</span>'
        f'<span style="background:{missing_color};color:#fff;padding:4px 10px;'
        f'border-radius:12px;font-size:12px;">Missing values: {s["missing"]}</span>'
        f'</div>'
        f'<div style="border-left:4px solid {PALETTE["muted"]};padding:8px 12px;'
        f'background:#F8F9FA;border-radius:4px;font-size:13px;color:#495057;">'
        f'{s["note"]}</div></div>'
    )

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(data=fig.data, layout=layout))
        display(HTML(info_html))

step_slider.observe(render, names="value")
display(VBox([step_slider, out]))
render()

---
## What's happening?

We applied every preprocessing technique from this folder in sequence:

| Step | Technique | Why | Notebook |
|---|---|---|---|
| 1 | Binary cabin indicator | 77% missing: extract presence/absence signal | `02_handling_missing_data` |
| 2 | Median/mode imputation | Fill remaining missing values | `02_handling_missing_data` |
| 3 | One-hot encoding | Convert sex and embarked to numeric | `03_encoding_categorical_variables` |
| 4 | Train-test split (stratified) | Honest evaluation, preserve survival rate | `06_train_test_split` |
| 5 | StandardScaler (train only) | Comparable scales; fit on train, transform both | `04_feature_scaling` |

The golden rule from `08_data_leakage.ipynb` is respected throughout: no test data was used in any fitting step.

---
## Save the clean files

In [ ]:
# Create the data directory if it does not exist
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
os.makedirs(DATA_DIR, exist_ok=True)

# Save the four split files
X_tr_final.to_csv(os.path.join(DATA_DIR, "X_train.csv"), index=False)
X_te_final.to_csv(os.path.join(DATA_DIR, "X_test.csv"),  index=False)
y_tr.to_csv(      os.path.join(DATA_DIR, "y_train.csv"), index=False, header=True)
y_te.to_csv(      os.path.join(DATA_DIR, "y_test.csv"),  index=False, header=True)

# Save the fitted scaler so the model folder can apply the same transformation
joblib.dump(scaler, os.path.join(DATA_DIR, "scaler.pkl"))

print("Saved to data/:")
for fname in ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv", "scaler.pkl"]:
    fpath = os.path.join(DATA_DIR, fname)
    size  = os.path.getsize(fpath)
    print(f"  {fname:<20}  {size:,} bytes")

print()
print("Training set shape:", X_tr_final.shape)
print("Test set shape:    ", X_te_final.shape)
print("Survival rate (train):", round(y_tr.mean(), 3), "  (test):", round(y_te.mean(), 3))

In [ ]:
# Verify the saved files load correctly and match what we saved
X_train_check = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"))
X_test_check  = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))
scaler_check  = joblib.load(os.path.join(DATA_DIR, "scaler.pkl"))

print("X_train loaded shape:", X_train_check.shape)
print("X_test loaded shape: ", X_test_check.shape)
print()
print("Numeric columns in training set (should be scaled, mean ≈ 0):")
print(X_train_check[["pclass", "age", "sibsp", "parch", "fare"]].describe().loc[["mean","std"]].round(3))
print()
print("Scaler is ready for the model folder:", type(scaler_check).__name__)

---
## Production note

> In production, this entire sequence (imputer, encoder, scaler) would be wrapped in a single `sklearn.Pipeline` that becomes a **model artifact**: stored alongside the trained model so it can transform new data the same way it transformed training data.
>
> You will build this Pipeline in your **model folder**, when you train your first algorithm. The preprocessing logic you have written here becomes the first stages of that Pipeline.

---
## Key takeaway

> **The preprocessing folder's job is done: raw Titanic data → clean, split, scaled files in `data/`. The model folder picks up from here.**

---
*Next up: 11_preprocessing_decisions_and_model_assumptions.ipynb. Which preprocessing steps you need depends on which algorithm you choose*